# Diffusion model for the 1D classical Heisenberg model

This notebook is an initial proof-of-concept for score-based diffusion on the O(3) Heisenberg model. I work at $L=16$, $\beta=3$, validate the heat-bath samples against the exact 1D transfer-matrix result, train a small U-Net score model, and test reverse-diffusion sampling.

The final checkpoint used for the results below is `uNet1D_20.pt`. The notebook assumes it is run from the repository root.

In [2]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

import model
import mcmc_1d_np as mcmc_np
import mcmc_1d_torch as mcmc
import rotation as rot
import sphere
import sde
import sde_np

beta = 3.
L = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def exactCorrelationTau(L, beta, tau = 0.0, ellMax = 5):
  expFactor = np.full(L, np.exp(-4*tau))
  expFactor[0] = 1.0

  return mcmc_np.exactCorrelation(L, beta, ellMax) * expFactor

def score(x, t, scoreNet):
    u = sde.uFromTau(t, sde.t0, sde.tF)

    with torch.no_grad():
      omega = scoreNet(x, u)
      return torch.cross(omega, x, dim=-1)

def score_np(x, t, scoreNet):
  """Score wrapper for the older NumPy SDE routines used during training diagnostics."""
  x_torch = torch.from_numpy(x).float().to(device)

  u_np = sde_np.uFromTau(t, sde.t0, sde.tF)
  u = torch.as_tensor(u_np, dtype=torch.float32, device=device)

  with torch.no_grad():
      F = scoreNet(x_torch, u)
      scoreOut = torch.cross(F, x_torch, dim=-1)

  return scoreOut.cpu().numpy()

## MCMC sampling and validation

The 1D model is convenient because the finite-volume correlation function and energy variance can be computed exactly from the transfer matrix. I use these as checks on the heat-bath samples before training the diffusion model.

In [25]:
trials = 1_000_000
burnIn = 10
filename = 'data/MCMC/cfgs1d_l{}_beta{}_trials{}_float32.npy'.format(L,beta,trials)

if os.path.exists(filename):
    print('load from file')
    cfgs = np.load(filename, allow_pickle=True)
else:
    print('file not found; generating samples')
    cfgs = mcmc_np.randomSpins(trials, L)
    cfgs = mcmc_np.heatBath(cfgs, beta, burnIn).astype(np.float32)
    os.makedirs('data/MCMC', exist_ok=True)
    np.save(filename, cfgs)

load from file


In [ ]:
# Compare sampled and exact correlation functions

corr = mcmc_np.correlation(cfgs)
corrExact = mcmc_np.exactCorrelation(L, beta)

for r in range(L):
    print(r, corr[r], corrExact[r])

0 1.0 1.0
1 0.6721343398094177 0.6724071777700129
2 0.45263928174972534 0.45338413028724145
3 0.3060935139656067 0.30745091935989133
4 0.20906990766525269 0.21102412551031283
5 0.14605014026165009 0.14854762902439766
6 0.10695469379425049 0.10995464630229376
7 0.08578377962112427 0.08903187509963231
8 0.07910050451755524 0.08241257386859094
9 0.08578377962112427 0.08903187509963231
10 0.10695469379425049 0.10995464630229376
11 0.14605014026165009 0.14854762902439766
12 0.20906990766525269 0.21102412551031283
13 0.3060935139656067 0.30745091935989133
14 0.45263928174972534 0.45338413028724145
15 0.6721343398094177 0.6724071777700129


In [ ]:
print("Sample energy variance:", mcmc_np.energyVariance(cfgs))

1.6358733


In [ ]:
# Exact result as the angular-momentum cutoff is increased

for ell in [1,2,3,4,5]:
    print(ell, mcmc_np.exactEnergyVariance(L, beta, ellMax=ell))

1 1.6394616463428378
2 1.6394645548104023
3 1.6394645548125908
4 1.6394645548125908
5 1.6394645548125908


### Diffusion sanity check

For independent heat-kernel diffusion of the spins, nonzero-separation correlations are suppressed by $e^{-4\tau}$. As a simple check, diffuse the equilibrium samples to $u=0.2$ and compare with this exact prediction.

In [ ]:
cfgs2 = sphere.hk(cfgs, sde.tauFromU(0.2, sde.t0, sde.tF))

print("Variance of energy at u = 0.2")
print(mcmc_np.energyVariance(cfgs2))

corr2 = mcmc_np.correlation(cfgs2)
corrExact2 = exactCorrelationTau(L, beta, tau = sde.tauFromU(0.2, sde.t0, sde.tF))

for r in range(L):
    print(r, corr2[r], corrExact2[r])

Variance of energy at u = 0.2 (1.639 at u = 0)
1.8684721
Correlation function at u = 0.2 (compared to u = 0)
0 1.0 1.0
1 0.6354013681411743 0.6356822882522665
2 0.4278794229030609 0.42862163124742075
3 0.28932884335517883 0.29065886029373494
4 0.19763913750648499 0.1994986124712536
5 0.13803794980049133 0.14043439727375426
6 0.10110092163085938 0.10394924902083316
7 0.08110450208187103 0.08416921764342332
8 0.0748177170753479 0.07791144305045394
9 0.08110449463129044 0.08416921764342332
10 0.10110092163085938 0.10394924902083316
11 0.13803794980049133 0.14043439727375426
12 0.1976391226053238 0.1994986124712536
13 0.28932884335517883 0.29065886029373494
14 0.4278794229030609 0.42862163124742075
15 0.6354013681411743 0.6356822882522665


## Training the U-Net

The network is trained by conditional score matching on heat-kernel-noised MCMC configurations. Diffusion times are sampled uniformly over $u\in[0.2,0.8]$, and each batch is augmented by a random global O(3) rotation.

In [27]:
uNet = model.UNet()
uNet.to(device)

numParameters = sum(p.numel() for p in uNet.parameters())
print(f"Number of parameters = {numParameters}")

Number of parameters = 114451


In [28]:
# Set up Adam
lr = 1e-4
optimizer = torch.optim.Adam(uNet.parameters(), lr=lr, betas=(0.9, 0.999))

# Set up DataLoader
batch_size = 1024
data_loader = DataLoader(cfgs, batch_size=batch_size, shuffle=True)

# Used for the physical diagnostic during training
tau2 = sde.tauFromU(np.linspace(1.,0.2,101))
os.makedirs('data/Models', exist_ok=True)

In [ ]:

n_epochs = 21

for i in range(n_epochs):

    lossTot = 0.0
    targetTot = 0.0
    predTot = 0.0
    crossTot = 0.0
    omegaTot = 0.0
    gradMax = 0.0
    nDiag = 0

    for step, x0 in enumerate(tqdm(data_loader), start=1):

        # rotate batch of vectors by a random rotation
        mat = rot.rotFromQ(rot.randomQ())
        x0 = torch.einsum('ij,...j->...i', mat, x0)

        # random diffusion time
        x0_np = x0.numpy()
        B = x0.shape[0]
        u_min = 0.2
        u_max = 0.8
        u_np = u_min + (u_max-u_min)*np.random.random((B,1,1))
        tau = sde.tauFromU(u_np, sde.t0, sde.tF)

        # diffuse clean configurations
        x_np = sphere.hk(x0_np, tau)

        # conditional score
        cosTh = sphere.dot(x0_np, x_np)
        tangent = x0_np - cosTh*x_np
        s_np = sphere.hkScore(cosTh, tau)*tangent

        # move to GPU
        u = torch.from_numpy(u_np).float().to(device)
        x = torch.from_numpy(x_np).float().to(device)
        target = torch.from_numpy(s_np).float().to(device)

        # network score
        omega = uNet(x, u)
        prediction = torch.cross(omega, x, dim=-1)

        # loss
        loss = torch.sum((prediction-target)**2, dim=-1).mean()

        # diagnostics before update
        with torch.no_grad():
            targetPower = torch.sum(target**2, dim=-1).mean()
            predPower = torch.sum(prediction**2, dim=-1).mean()
            cross = torch.sum(prediction*target, dim=-1).mean()
            omegaPower = torch.sum(omega**2, dim=-1).mean()

        lossTot += loss.item()
        targetTot += targetPower.item()
        predTot += predPower.item()
        crossTot += cross.item()
        omegaTot += omegaPower.item()
        nDiag += 1

        # gradient update
        optimizer.zero_grad()
        loss.backward()
        gradNorm = torch.nn.utils.clip_grad_norm_(uNet.parameters(), max_norm=1.0)
        optimizer.step()

        g = gradNorm.item()
        gradMax = max(gradMax, g)

    # training diagnostics
    lossAvg = lossTot/nDiag
    targetAvg = targetTot/nDiag
    predAvg = predTot/nDiag
    crossAvg = crossTot/nDiag
    omegaAvg = omegaTot/nDiag
    gain = 2*crossAvg - predAvg

    print(
        f"loss {lossAvg:.6g}   "
        f"target {targetAvg:.6g}   "
        f"pred {predAvg:.6g}   "
        f"omega {omegaAvg:.6g}   "
        f"gain {gain:.6g}   "
        f"grad max {gradMax:.6g}"
    )

    # physical diagnostic
    samples = mcmc_np.randomSpins(10000, L)

    for j in tqdm(range(100)):
        dt = tau2[j] - tau2[j+1]
        samples = sde.eulerMaruyamaStep(
            samples,
            tau2[j],
            dt,
            lambda x, t: score_np(x, t, uNet)
        )

    enVar = mcmc_np.energyVariance(samples)
    corr = mcmc_np.correlation(samples)

    print(
        f"Energy, Energy Var: ({corr[1]}, {enVar});   "
        f"Training data: (0.6354, 1.8651)"
    )

    if i % 5 == 0:
        torch.save(uNet.state_dict(), f'data/Models/uNet1D_{i}.pt')
        print(f"Saved: data/Models/uNet1D_{i}.pt")

        for r in range(L):
            print(r, corr[r], corr2[r])

## Inference

Reload the final saved checkpoint before generating samples. The intermediate-time test at $u=0.6$ stays inside the range used for training; the final tests integrate the reverse SDE all the way to $u=0$.

In [3]:
uNet = model.UNet().to(device)
uNet.load_state_dict(torch.load("data/Models/uNet1D_20.pt", map_location=device))

<All keys matched successfully>

In [ ]:
tau6 = sde.tauFromU(torch.linspace(1.,0.6,101))

In [16]:
# Intermediate-time check at u=0.6

samplesF = mcmc.randomSpins(10000, L, device=device)
samples = samplesF
for i in tqdm(range(len(tau6)-1)):
    dt = tau6[i]-tau6[i+1]
    samples = sde.eulerMaruyamaStep(samples, tau6[i], dt, lambda x, t : score(x, t, uNet))

corr = mcmc.correlation(samples)
tau = sde.tauFromU(0.6)
corrExact6 = exactCorrelationTau(L, beta, tau=tau.item())

for r in range(L):
    print(r, corr[r], corrExact6[r])

0 tensor(1., device='cuda:0') 1.0
1 tensor(0.2188, device='cuda:0') 0.22729586554105824
2 tensor(0.1437, device='cuda:0') 0.15325883143898564
3 tensor(0.0945, device='cuda:0') 0.10392857949413924
4 tensor(0.0603, device='cuda:0') 0.07133313391594573
5 tensor(0.0399, device='cuda:0') 0.05021401173192275
6 tensor(0.0274, device='cuda:0') 0.03716830713263
7 tensor(0.0220, device='cuda:0') 0.030095718458311792
8 tensor(0.0187, device='cuda:0') 0.027858175712893406
9 tensor(0.0220, device='cuda:0') 0.030095718458311792
10 tensor(0.0274, device='cuda:0') 0.03716830713263
11 tensor(0.0399, device='cuda:0') 0.05021401173192275
12 tensor(0.0603, device='cuda:0') 0.07133313391594573
13 tensor(0.0945, device='cuda:0') 0.10392857949413924
14 tensor(0.1437, device='cuda:0') 0.15325883143898564
15 tensor(0.2188, device='cuda:0') 0.22729586554105824


### Full reverse diffusion

The following is the main endpoint test: 40,000 samples evolved from the nearly uniform endpoint to $u=0$ with 400 Euler-Maruyama steps.

In [19]:
tau0 = sde.tauFromU(torch.linspace(1.0,0.0,401))

samplesF = mcmc.randomSpins(40000, L, device=device)
samples = samplesF
for i in tqdm(range(len(tau0)-1)):
    dt = tau0[i]-tau0[i+1]
    samples = sde.eulerMaruyamaStep(samples, tau0[i], dt, lambda x, t : score(x, t, uNet))

corr = mcmc.correlation(samples)
corrExact = mcmc_np.exactCorrelation(L, beta, ellMax=5)

for r in range(L):
    print(r, corr[r], corrExact[r])

0 tensor(1., device='cuda:0') 1.0
1 tensor(0.6630, device='cuda:0') 0.6724071777700129
2 tensor(0.4402, device='cuda:0') 0.45338413028724145
3 tensor(0.2910, device='cuda:0') 0.30745091935989133
4 tensor(0.1935, device='cuda:0') 0.21102412551031283
5 tensor(0.1318, device='cuda:0') 0.14854762902439766
6 tensor(0.0939, device='cuda:0') 0.10995464630229376
7 tensor(0.0741, device='cuda:0') 0.08903187509963231
8 tensor(0.0678, device='cuda:0') 0.08241257386859092
9 tensor(0.0741, device='cuda:0') 0.08903187509963231
10 tensor(0.0939, device='cuda:0') 0.10995464630229376
11 tensor(0.1318, device='cuda:0') 0.14854762902439766
12 tensor(0.1935, device='cuda:0') 0.21102412551031283
13 tensor(0.2910, device='cuda:0') 0.30745091935989133
14 tensor(0.4402, device='cuda:0') 0.45338413028724145
15 tensor(0.6630, device='cuda:0') 0.6724071777700129


A simple predictor-corrector variant, using RK2 together with Langevin updates near the middle of the trajectory, gives a modest improvement.

In [20]:
tauA = sde.tauFromU(np.linspace(1.0,0.5,11))
tauB = sde.tauFromU(np.linspace(0.5,0.0,11))

samplesF = mcmc.randomSpins(40000, L, device=device)
samples = samplesF
for i in tqdm(range(len(tauA)-1)):
    dt = tauA[i]-tauA[i+1]
    samples = sde.rk2Step(samples, tauA[i], dt, lambda x, t : score(x, t, uNet))

for i in tqdm(range(360)):
    samples = sde.langevinStep(samples, tauB[0], .005, lambda x, t : score(x, t, uNet))

for i in tqdm(range(len(tauB)-1)):
    dt = tauB[i]-tauB[i+1]
    samples = sde.rk2Step(samples, tauB[i], dt, lambda x, t : score(x, t, uNet))

corr = mcmc.correlation(samples)

for r in range(L):
    print(r, corr[r], corrExact[r])

0 tensor(1., device='cuda:0') 1.0
1 tensor(0.6679, device='cuda:0') 0.6724071777700129
2 tensor(0.4472, device='cuda:0') 0.45338413028724145
3 tensor(0.3010, device='cuda:0') 0.30745091935989133
4 tensor(0.2048, device='cuda:0') 0.21102412551031283
5 tensor(0.1425, device='cuda:0') 0.14854762902439766
6 tensor(0.1041, device='cuda:0') 0.10995464630229376
7 tensor(0.0831, device='cuda:0') 0.08903187509963231
8 tensor(0.0765, device='cuda:0') 0.08241257386859092
9 tensor(0.0831, device='cuda:0') 0.08903187509963231
10 tensor(0.1041, device='cuda:0') 0.10995464630229376
11 tensor(0.1425, device='cuda:0') 0.14854762902439766
12 tensor(0.2048, device='cuda:0') 0.21102412551031283
13 tensor(0.3010, device='cuda:0') 0.30745091935989133
14 tensor(0.4472, device='cuda:0') 0.45338413028724145
15 tensor(0.6679, device='cuda:0') 0.6724071777700129


The 1D model was mainly used to establish that the score model and manifold-valued SDE machinery worked. The subsequent work moved to the 2D O(3) model rather than further optimizing this 1D example.